# Task 2 — California Housing Value Prediction
**Course:** Machine Learning Fundamentals (AIC354)  
**Task:** Evaluate five regression algorithms on the full Kaggle dataset using 10-fold cross-validation.

> **Deployment link:** Add your live Hugging Face Spaces URL here after deployment.

## Problem statement
Predict the median house value for a California district from census-related features. The target column is `median_house_value`.

**Dataset:** [California Housing Prices — Kaggle](https://www.kaggle.com/datasets/camnugent/california-housing-prices)  
Download the dataset and upload the CSV named `housing.csv` when prompted below.

### Algorithms selected
1. Extra Trees Regressor
2. AdaBoost Regressor
3. Elastic Net Regression
4. Bayesian Ridge Regression
5. Huber Regression

These five algorithms do not overlap with the instructor's Car Price Prediction notebooks, which use Linear Regression, K-Nearest Neighbors, Decision Tree, Random Forest, and Gradient Boosting.

We will report RMSE, MAE, R², and runtime. Lower RMSE/MAE is better; higher R² is better.

## 1. Import libraries

In [ ]:
import os, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor
from sklearn.linear_model import ElasticNet, BayesianRidge, HuberRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import clone

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Upload and load the FULL Kaggle dataset

In [ ]:
uploaded = files.upload()  # Choose housing.csv downloaded from Kaggle
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Please upload the Kaggle CSV file.")
df = pd.read_csv(csv_files[0])

print("Dataset file:", csv_files[0])
print("Dataset shape:", df.shape)
display(df.head())

## 3. Inspect and clean the data

In [ ]:
print("Column names:", df.columns.tolist())
display(df.info())
display(df.describe(include="all").T)
print("\\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_values"))
print("\\nDuplicate rows:", df.duplicated().sum())

TARGET = "median_house_value"
if TARGET not in df.columns:
    raise ValueError(f"Expected target column '{TARGET}'. Check that you uploaded the California Housing Prices housing.csv file.")

# Remove duplicate records, if any. Do not remove rows just because a feature is missing;
# missing values are imputed inside each training fold to prevent data leakage.
df = df.drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=[TARGET])
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(float)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print(f"Rows used: {len(df):,}")
print(f"Input features: {X.shape[1]} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")
print(f"Target: {TARGET}")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(y, bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Median House Value")
axes[0].set_xlabel("Median house value")
sns.boxplot(x=y, ax=axes[1])
axes[1].set_title("Target Value — Boxplot")
axes[1].set_xlabel("Median house value")
plt.tight_layout()
plt.show()

In [ ]:
# Missing-value chart
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
plt.figure(figsize=(10, 4))
if len(missing):
    sns.barplot(x=missing.values, y=missing.index)
    plt.title("Missing Values by Column")
    plt.xlabel("Number of missing values")
else:
    plt.text(0.5, 0.5, "No missing values found", ha="center", va="center")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical variables only
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(11, 8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap — Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Income vs target and geographic distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if "median_income" in X.columns:
    sample = df.sample(min(5000, len(df)), random_state=42)
    sns.scatterplot(data=sample, x="median_income", y=TARGET, alpha=0.35, s=18, ax=axes[0])
    axes[0].set_title("Median Income vs House Value")
else:
    axes[0].text(0.5, 0.5, "median_income not found", ha="center")
if {"longitude", "latitude"}.issubset(X.columns):
    sample = df.sample(min(5000, len(df)), random_state=42)
    points = axes[1].scatter(sample["longitude"], sample["latitude"], c=sample[TARGET], s=8, alpha=0.6)
    axes[1].set_title("Geographic Pattern of House Values")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    plt.colorbar(points, ax=axes[1], label="Median house value")
else:
    axes[1].text(0.5, 0.5, "Latitude/longitude not found", ha="center")
plt.tight_layout()
plt.show()

## 5. Preprocessing and model definitions

Preprocessing is placed inside a scikit-learn pipeline. This matters because imputing, encoding, or scaling before cross-validation can leak information from validation folds into training folds. Each fold learns its preprocessing only from that fold's training portion.

In [ ]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
], remainder="drop")

models = {
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100, random_state=42, n_jobs=-1, min_samples_leaf=1
    ),
    "AdaBoost": AdaBoostRegressor(
        n_estimators=80, learning_rate=0.5, random_state=42
    ),
    "Elastic Net": ElasticNet(
        alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=42
    ),
    "Bayesian Ridge": BayesianRidge(),
    "Huber Regression": HuberRegressor(
        epsilon=1.35, alpha=0.0001, max_iter=300
    )
}
print("Number of algorithms:", len(models))
for name in models:
    print("-", name)

## 6. Evaluate all five algorithms using 10-fold cross-validation

In [ ]:
# Shuffle ensures folds are not simply consecutive sections of the CSV.
cv = KFold(n_splits=10, shuffle=True, random_state=42)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

results = []
for name, regressor in models.items():
    print(f"Evaluating {name} with 10-fold CV...")
    pipeline = Pipeline(steps=[
        ("preprocessing", clone(preprocessor)),
        ("model", clone(regressor))
    ])
    start = time.time()
    scores = cross_validate(
        pipeline, X, y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,          # safer for mixed estimators on Colab
        return_train_score=False,
        error_score="raise"
    )
    elapsed = time.time() - start
    fold_rmse = -scores["test_rmse"]
    fold_mae = -scores["test_mae"]
    fold_r2 = scores["test_r2"]
    results.append({
        "Model": name,
        "Mean RMSE": fold_rmse.mean(),
        "RMSE Std": fold_rmse.std(ddof=1),
        "Mean MAE": fold_mae.mean(),
        "Mean R²": fold_r2.mean(),
        "R² Std": fold_r2.std(ddof=1),
        "CV Runtime (seconds)": elapsed
    })

results_df = pd.DataFrame(results).sort_values("Mean RMSE").reset_index(drop=True)
display(results_df.round(4))
print("Best model by mean 10-fold RMSE:", results_df.loc[0, "Model"])

## 7. Visual comparison of the five algorithms

In [ ]:
plt.figure(figsize=(11, 5))
plot_df = results_df.sort_values("Mean RMSE", ascending=True)
sns.barplot(data=plot_df, x="Mean RMSE", y="Model")
plt.title("10-Fold Cross-Validation: RMSE Comparison (lower is better)")
plt.xlabel("Mean RMSE")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
plot_df = results_df.sort_values("Mean MAE", ascending=True)
sns.barplot(data=plot_df, x="Mean MAE", y="Model")
plt.title("10-Fold Cross-Validation: MAE Comparison (lower is better)")
plt.xlabel("Mean MAE")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
plot_df = results_df.sort_values("Mean R²", ascending=False)
sns.barplot(data=plot_df, x="Mean R²", y="Model")
plt.title("10-Fold Cross-Validation: R² Comparison (higher is better)")
plt.xlabel("Mean R²")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Display per-fold RMSE stability for each algorithm
fold_rows = []
for name, regressor in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessing", clone(preprocessor)),
        ("model", clone(regressor))
    ])
    scores = cross_validate(
        pipeline, X, y, cv=cv,
        scoring={"rmse": "neg_root_mean_squared_error"},
        n_jobs=1, return_train_score=False
    )
    for fold_number, score in enumerate(-scores["test_rmse"], start=1):
        fold_rows.append({"Model": name, "Fold": fold_number, "RMSE": score})
fold_df = pd.DataFrame(fold_rows)

plt.figure(figsize=(12, 6))
sns.lineplot(data=fold_df, x="Fold", y="RMSE", hue="Model", marker="o")
plt.title("RMSE Across the 10 Folds")
plt.xticks(range(1, 11))
plt.tight_layout()
plt.show()

## 8. Train the selected model on the full dataset and save it

The model selected below is the one with the lowest **mean 10-fold RMSE**. Cross-validation is used for model comparison; after selecting the model, we fit the complete pipeline on all available rows for deployment.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_pipeline = Pipeline(steps=[
    ("preprocessing", clone(preprocessor)),
    ("model", clone(models[best_name]))
])
best_pipeline.fit(X, y)

MODEL_FILE = "california_housing_model.joblib"
joblib.dump({
    "pipeline": best_pipeline,
    "model_name": best_name,
    "feature_columns": X.columns.tolist(),
    "target": TARGET,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}, MODEL_FILE)

results_df.to_csv("cross_validation_results.csv", index=False)
df.to_csv("housing_cleaned.csv", index=False)
print("Selected model:", best_name)
print("Saved:", MODEL_FILE, "and cross_validation_results.csv")
files.download(MODEL_FILE)
files.download("cross_validation_results.csv")

## 9. Prediction demonstration

This uses one existing row as an example input. It is only a demonstration of the pipeline's prediction function, not an independent test score.

In [ ]:
example_X = X.iloc[[0]].copy()
actual_value = float(y.iloc[0])
predicted_value = float(best_pipeline.predict(example_X)[0])
print(f"Example actual value:    ${actual_value:,.2f}")
print(f"Example predicted value: ${predicted_value:,.2f}")

## 10. Results interpretation and conclusion

After running all cells, write your conclusion using the actual values in `results_df`:

- **Best model:** the model with the lowest mean 10-fold RMSE.
- **RMSE:** typical prediction error in the target's units; lower is better.
- **MAE:** average absolute prediction error; lower is better.
- **R²:** proportion of target variation explained by the model; higher is better.
- **Stability:** compare the RMSE standard deviation and fold chart. Lower variation indicates more consistent performance across folds.
- **Runtime:** consider both accuracy and computational cost.

Do not enter invented metric values. Use the results generated by your Colab runtime. The dataset describes California districts based on 1990 census data, so predictions should not be presented as current market appraisals.